## Subagents with the Claude Managed Agents API

### Installing Utilities and Libraries

In [1]:
%pip install anthropic==0.120.2 python-dotenv==1.2.2

Looking in indexes: https://pypi.org/simple/

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: /usr/local/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Setting up the Environment

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

claude_model_name = os.getenv("CLAUDE_MODEL_NAME")
claude_api_key = os.getenv("CLAUDE_API_KEY")

### Creating the Anthropic Client

In [3]:
import anthropic

client = anthropic.Anthropic(api_key = claude_api_key)

### Create the Researcher Agent

In [4]:
researcher_agent = client.beta.agents.create(
    name="Researcher-Agent",
    model=claude_model_name,
    system="""You are a knowledgeable researcher. Your task is to gather information and provide insights on a given topic.
              You should use reliable sources and present the information in a clear and concise manner.""",
    tools=[
        {"type": "agent_toolset_20260401"},
    ],
)

print(f"Agent ID: {researcher_agent.id}, version: {researcher_agent.version}")

Agent ID: agent_01B92AVwwqeuKCnU1EjCuuFT, version: 1


### Create the Writer Agent

In [5]:
writer_agent = client.beta.agents.create(
    name="Writer-Agent",
    model=claude_model_name,
    system="""You are a creative writer. Your task is to write an essay on a given topic.
              You should focus on clarity, coherence, and engaging storytelling""",
)

print(f"Agent ID: {writer_agent.id}, version: {writer_agent.version}")

Agent ID: agent_016CY837YzRxWnsLJU1z4c9N, version: 1


### Create the Coordinator Agent

In [6]:
coordinator_agent = client.beta.agents.create(
    name="Editorial-Head",
    model=claude_model_name,
    system="""You are the Editorial Head Agent which whill coordinate and delegate work to the 
              researcher agent for researching topics and the writer agent for synthesizing the topics researched
              in a refined output which could be an article, essay etc. """,
    tools=[
        {"type": "agent_toolset_20260401"},
    ],
    multiagent={
        "type": "coordinator",
        "agents": [
            {"type": "agent", "id": researcher_agent.id},
            {"type": "agent", "id": writer_agent.id}
        ]
    }
)

print(f"Agent ID: {coordinator_agent.id}, version: {coordinator_agent.version}")

Agent ID: agent_01LmfSuHjp8KkxcepV6mThRB, version: 1


### Create the Environment

In [7]:
environment = client.beta.environments.create(
    name="quickstart-env",
    config={
        "type": "cloud",
        "networking": {"type": "unrestricted"},
    },
)

print(f"Environment ID: {environment.id}")

Environment ID: env_01KWyQPPw1Zw9mb6wbwcAgY5


### Create the Session

In [8]:
session = client.beta.sessions.create(
    agent=coordinator_agent.id,
    environment_id=environment.id,
)

### Execute the Agent

In [9]:
with client.beta.sessions.events.stream(session.id) as stream:
            # Send the user message after the stream opens
            client.beta.sessions.events.send(
                session.id,
                events=[
                    {
                        "type": "user.message",
                        "content": [
                            {
                                "type": "text",
                                "text": """write an essay about the impact of AI on society
                                           Don't perform a lot of research, use only one instance of researcher agent
                                           and rather make it quick pls. Keep the essay really short under 250 words pls""",
                            },
                        ],
                    },
                ],
            )

            # Process streaming events
            for event in stream:
                match event.type:
                    case "agent.message":
                        for block in event.content:
                            print(block.text, end="")
                    case "agent.tool_use":
                        print(f"\n[Using tool: {event.name}]")
                    case "session.status_idle":
                        print("\n\nAgent finished.")
                        break

Still working — I'll wait for the researcher's report before bringing in the writer.The writer agent is drafting the essay now — I'll share it as soon as it comes back.Here's the essay on the impact of AI on society (238 words):

---

**The Impact of AI on Society**

Artificial intelligence has moved from speculative fiction to a defining force reshaping modern life, touching nearly every sector while raising profound questions about how societies should adapt.

Economically, AI automates routine cognitive and manual tasks, displacing workers in data entry, customer service, and basic coding, even as it creates new roles like AI trainers and ethics specialists. The result is real productivity growth, but its benefits are unevenly distributed, deepening inequality between digitally skilled workers and those left behind, fueling debates over reskilling and universal basic income.

Education and healthcare reveal similar dualities. Adaptive learning platforms personalize instruction at sc